# 📘 Notebook 12: The Cloud and an End-to-End Pipeline

### Course: *From One Table to a Cluster: Working with Data at Scale*
**Instructor:** Ioannis Tsioulis

*Module D: Data in Motion and in the Cloud · Notebook 2 of 2 in this module · (12 of 12 overall)*

---

## 🎯 Learning objectives

By the end of this notebook you will be able to:

- Say what "the cloud" is in concrete terms, and inspect the rented machine you are working on
- Use **Amdahl's law** to predict how much faster a job becomes with more machines, and where it stops helping
- Build a simple **cost model** and find where one way of buying computing becomes cheaper than another
- Assemble a complete **pipeline** from raw events to a report, using the techniques of the whole course
- Make a pipeline **idempotent** and **incremental**, and make it **check** its own work
- Protect personal data in a pipeline by **pseudonymising** it, and carry out a request to **erase** it

> **Prerequisites:** the whole course. This notebook uses SQL and indexes (Notebooks 01 to 03), the star schema (04), Parquet and partitioning (05), nested records (06), streaming through files (07), and deduplication and idempotence (10 and 11).
>
> 🔭 **Why this notebook matters:** eleven notebooks have each taught one technique. Real work is never one technique. It is a chain of them, running unattended every night, on machines somebody is paying for by the hour, handling information about real people. This notebook builds such a chain from end to end, small enough to read in one sitting and complete enough to show how the parts fit. It then asks the two questions that engineers are asked most often and textbooks answer least: *how much will it cost*, and *what happens when a customer asks you to delete their data*.

> 📦 **Libraries used in this notebook.**
>
> - **PyArrow** (Notebook 05) for Parquet files, and **SQLite** (Notebook 01) for the warehouse.
> - **psutil**, a small library that reports the processor, memory and disk of the machine a program is running on.
> - **Matplotlib**, for two charts.
>
> The cell below installs what is needed. In Google Colab these libraries are already present and the cell finishes in a moment.

In [ ]:
%pip install -q pyarrow psutil matplotlib

In [ ]:
import datetime
import hashlib
import json
import os
import platform
import random
import shutil
import sqlite3
import tempfile
import time

import matplotlib.pyplot as plt
import psutil
import pyarrow as pa
import pyarrow.parquet as pq

## 1. Somebody else's computer

### Intuition first
There is an old joke that "the cloud" is just somebody else's computer. It is very nearly the whole truth. A cloud provider owns warehouses full of machines, and **rents** them out by the hour or by the second, along with storage and ready-made services built on top. You do not buy a server, wait for it to be delivered, and find it too small a year later. You ask for one, or a thousand, use them, and hand them back.

If you are reading this in Google Colab, you are using one now. Let us look at it.

In [ ]:
memory = psutil.virtual_memory()
disk = shutil.disk_usage(tempfile.gettempdir())

print("Operating system:", platform.system(), platform.release())
print("Processor cores: ", os.cpu_count())
print(f"Memory:           {memory.total / 1e9:.1f} GB in total, {memory.available / 1e9:.1f} GB available")
print(f"Disk:             {disk.total / 1e9:.0f} GB in total, {disk.free / 1e9:.0f} GB free")

In Colab, those figures describe a virtual machine in one of Google's data centres, created for you a few minutes ago and due to be destroyed when you close the notebook. Everything you have run in this course ran there.

That is the essential service. Three properties make it different from owning hardware, and they shape every decision in this notebook.

- **Elasticity.** Capacity can be added and removed in minutes. A job that needs a hundred machines for one hour a night can have exactly that.
- **Pay for what you use.** Renting a hundred machines for one hour costs about the same as renting one machine for a hundred hours.
- **Storage and computing are separate.** Data sits cheaply in a storage service, and computing is attached to it only while a job runs. This is why the files of Notebook 05, and not a database server, are the usual home of large data in the cloud.

The second property contains a remarkable promise: with a hundred machines the job should finish a hundred times sooner, for the same money. The next section examines how far that promise holds.

## 2. How much faster? Amdahl's law

### Intuition first
One cook takes sixty minutes to prepare a dinner. Fifty of those minutes are chopping, which several cooks could share. Ten are the roast in the oven, which takes ten minutes however many cooks stand around it.

With two cooks: 25 minutes of chopping plus 10 of roasting, 35 in all. With five: 10 plus 10, which is 20. With fifty cooks: 1 plus 10, which is 11. With a thousand cooks the dinner still takes just over ten minutes. The part that cannot be shared sets a limit that no amount of help can break.

### Formal definition
Let $p$ be the fraction of a job that can be done in parallel, and $1 - p$ the fraction that must be done in sequence. With $n$ workers, the job runs faster by a factor of

$$\text{speedup}(n) = \frac{1}{(1 - p) + \dfrac{p}{n}}$$

As $n$ grows, $p/n$ shrinks towards zero and the speedup approaches $\dfrac{1}{1 - p}$, and never passes it. This is **Amdahl's law**.

In [ ]:
def speedup(parallel_fraction, workers):
    return 1 / ((1 - parallel_fraction) + parallel_fraction / workers)

print("The dinner (50 of 60 minutes can be shared):")
for cooks in [1, 2, 5, 50, 1000]:
    print(f"  {cooks:>5} cooks: {60 / speedup(50 / 60, cooks):5.1f} minutes")

In [ ]:
workers = [1, 2, 4, 8, 16, 32, 64, 128, 256, 512, 1024]

plt.figure(figsize=(7, 4))
for fraction in [0.50, 0.90, 0.95, 0.99]:
    plt.plot(workers, [speedup(fraction, n) for n in workers], marker="o", label=f"{fraction:.0%} parallel (limit {1 / (1 - fraction):.0f}x)")
plt.xscale("log", base=2)
plt.title("Amdahl's law: speedup against number of workers")
plt.xlabel("Workers (log scale)")
plt.ylabel("Times faster than one worker")
plt.legend()
plt.grid(True)
plt.show()

print(f"{'parallel part':>13} {'16 workers':>11} {'256 workers':>12} {'limit':>7}")
for fraction in [0.50, 0.90, 0.95, 0.99]:
    print(f"{fraction:>13.0%} {speedup(fraction, 16):>10.1f}x {speedup(fraction, 256):>11.1f}x {1 / (1 - fraction):>6.0f}x")

Read the table. A job that is 90% parallel, which sounds like a great deal, can never be made more than ten times faster. With 256 machines it runs 9.7 times faster than with one: the last 240 machines contributed almost nothing, and each was paid for.

Where does the sequential part of a real data job come from? You have met every source of it already: reading a single file that cannot be split, the driver program that plans the work and collects the results, a shuffle in which everything waits for the slowest partition, the one heavy key of a skewed dataset (Notebook 08), and a final step that writes a single output.

The practical lessons are two. Before adding machines, **find the sequential part and shrink it**, since that is what sets the ceiling. And **measure** the speedup you actually get: if doubling the machines no longer nearly halves the time, you have reached the flat part of the curve, and further machines are money spent on nothing.

## 3. What will it cost?

Engineers are asked this constantly, and most find it harder than any algorithm. The reasoning is simple once the pieces are named. A cloud bill has three main parts.

- **Computing**: the number of processor cores, times the hours they are held, times a price per core-hour. The price is lower if you commit to keeping the machine for a long time.
- **Storage**: the gigabytes kept, times the months they are kept, times a price per gigabyte-month.
- **Scanning**, in services that charge by the query: the gigabytes a query reads, times a price.

The prices in the cell below are **round numbers invented for this exercise**. Real prices differ between providers and regions, and change. Look them up before any real decision. The reasoning does not change.

In [ ]:
PRICE_PER_CORE_HOUR = 0.05            # illustrative figures, not real prices
PRICE_PER_RESERVED_CORE_HOUR = 0.025  # half price, for a machine rented for a year or more
PRICE_PER_GB_MONTH = 0.02
PRICE_PER_TB_SCANNED = 5.00

def always_on(cores):
    """A machine rented around the clock for a long period, whether or not it is busy."""
    return cores * 24 * 30 * PRICE_PER_RESERVED_CORE_HOUR

def rented_nightly(core_hours_per_night):
    """A cluster created each night for one job, and released when it finishes."""
    return core_hours_per_night * 30 * PRICE_PER_CORE_HOUR

machine_cores = 16
capacity = machine_cores * 24                                  # the most work one such machine can do in a day

print(f"A {machine_cores}-core machine, always on: {always_on(machine_cores):,.0f} a month, whatever it does\n")
print(f"{'nightly work':>20} {'always-on machine':>19} {'cluster by the hour':>21}   cheaper")
for core_hours in [5, 20, 80, 320, 640]:
    fixed = always_on(machine_cores) if core_hours <= capacity else None
    rented = rented_nightly(core_hours)
    fixed_text = f"{fixed:,.0f}" if fixed is not None else "cannot finish in a day"
    if fixed is None or rented < fixed:
        cheaper = "cluster by the hour"
    else:
        cheaper = "always-on machine"
    print(f"{core_hours:>9} core-hours {fixed_text:>19} {rented:>21,.0f}   {cheaper}")

The always-on machine costs the same every month whether it works for ten minutes a night or for all twenty-four hours. Providers charge less per hour for a machine you commit to for a long period, here half. The rented cluster costs the full hourly price, in proportion to the work.

For light work the cluster is far cheaper, since the permanent machine would stand idle nearly all day. Once the nightly work is heavy enough to keep the permanent machine busy most of the time, the positions reverse. And beyond what one machine can do in a day, it is no longer an option at all.

| | Light or occasional work | Heavy, steady work |
|---|---|---|
| A machine rented permanently | paid for while it sits idle | fully used, and cheap per unit of work |
| Machines rented by the hour | you pay only for the minutes used | you pay for every hour, with no discount for loyalty |
| Better choice | **rent by the hour** | **rent permanently** (or buy) |

The quantity that decides is **utilisation**: the share of the time a machine is actually working. A machine used 5% of the time is twenty times more expensive, per unit of work, than the same machine used all the time. Elasticity is valuable exactly when the load is **uneven**: a nightly job, a monthly report, a shop on the day of a sale. For a load that is flat around the clock it brings nothing.

### The format is part of the bill
Many cloud services charge for analysis by the amount of data a query **reads**. That makes the lessons of Notebook 05 into money. Suppose a table of one terabyte as CSV, queried a hundred times a day for two of its twenty columns.

In [ ]:
def monthly_scan_cost(table_gb, share_read, queries_per_day):
    return table_gb / 1000 * share_read * queries_per_day * 30 * PRICE_PER_TB_SCANNED

csv_gb = 1000
parquet_gb = csv_gb / 6                         # the compression ratio we measured in Notebook 05 was of this order

designs = [
    ("CSV: every query reads the whole file", csv_gb, 1.0),
    ("Parquet: only 2 of 20 columns are read", parquet_gb, 2 / 20),
    ("Parquet, partitioned by day: one month of two years", parquet_gb, 2 / 20 / 24),
]
print(f"{'design':<54} {'storage':>9} {'queries':>11} {'total a month':>14}")
for name, size_gb, share_read in designs:
    storage = size_gb * PRICE_PER_GB_MONTH
    queries = monthly_scan_cost(size_gb, share_read, queries_per_day=100)
    print(f"{name:<54} {storage:>9,.0f} {queries:>11,.0f} {storage + queries:>14,.0f}")

The same data and the same questions, and a bill that differs by a factor of a thousand, entirely because of how the bytes were laid out. Column pruning and partitioning are not refinements for specialists. In a system that charges by the byte read, they are the largest item on the invoice.

> ⚠️ **Common pitfalls**
>
> - Leaving a cluster running after the job has finished. Nothing in the cloud switches itself off out of thrift, and idle machines are the commonest cause of a surprising bill.
> - Estimating only the computing. Storage grows every day and is never deleted, and moving data **out** of a provider's network is often charged separately.
> - Assuming that more machines means sooner. Section 2 sets the limit. Beyond it, cost rises and time does not fall.
> - Choosing a design by its price on the first day. Ask what it costs when the data is ten times larger.

## 4. The whole course in one pipeline

Now we build something. Each night the bookshop's website delivers a file of the day's orders. By morning, the managers want to see revenue by day and by category. Between the two lies everything this course has covered.

```
 raw files            validate            store               load                 report
 (JSON lines,   ──►   and clean    ──►    as Parquet,   ──►   into a star    ──►   with SQL
  messy)              (Nb 06, 11)         partitioned         schema               (Nb 02)
                                          by day (Nb 05)      (Nb 04, 03)

                 at every step: count, check, and be safe to run twice (Nb 10, 11)
```

### The raw data
The orders arrive as **JSON lines**: one nested document per line, as in Notebook 06. And they arrive dirty, as real data does: some orders are sent twice, some have an impossible quantity, some lack a customer, and now and then a line is cut off halfway.

In [ ]:
workspace = tempfile.mkdtemp()
raw_folder = os.path.join(workspace, "raw")
lake_folder = os.path.join(workspace, "lake")
os.makedirs(raw_folder)

CITIES = ["Athens", "Thessaloniki", "Patras", "Heraklion", "Larissa", "Volos", "Ioannina", "Chania"]
CATEGORIES = ["Fiction", "Science", "History", "Children", "Cooking", "Travel", "Art", "Computing"]
catalogue_rng = random.Random(0)
CATALOGUE = [(f"Book {i}", catalogue_rng.choice(CATEGORIES), round(catalogue_rng.uniform(5, 60), 2)) for i in range(1, 201)]

def write_raw_day(day, orders=3000):
    rng = random.Random(day.toordinal())
    path = os.path.join(raw_folder, f"orders_{day.isoformat()}.jsonl")
    with open(path, "w", encoding="utf-8") as file:
        for number in range(orders):
            customer_id = rng.randint(1, 2000)
            order = {
                "order_id": f"{day.strftime('%Y%m%d')}-{number:05d}",
                "time": f"{day.isoformat()}T{rng.randint(0, 23):02d}:{rng.randint(0, 59):02d}:00",
                "customer": {"id": customer_id, "email": f"customer{customer_id}@example.com", "city": CITIES[customer_id % 8]},
                "lines": [],
            }
            for line in range(rng.randint(1, 4)):
                title, category, price = rng.choice(CATALOGUE)
                order["lines"].append({"title": title, "category": category, "price": price, "quantity": rng.randint(1, 3)})

            roll = rng.random()
            if roll < 0.010:
                del order["customer"]                              # no customer
            elif roll < 0.020:
                order["lines"][0]["quantity"] = -1                 # an impossible quantity
            text = json.dumps(order)
            if roll > 0.995:
                text = text[:len(text) // 2]                       # a line cut off halfway
            file.write(text + "\n")
            if 0.970 < roll < 0.990:
                file.write(text + "\n")                            # the same order sent twice
    return path

days = [datetime.date(2024, 3, 1) + datetime.timedelta(days=i) for i in range(5)]
raw_paths = {day: write_raw_day(day) for day in days}

with open(raw_paths[days[0]], encoding="utf-8") as file:
    print(json.dumps(json.loads(file.readline()), indent=1)[:520], "...")
print(f"\n{len(raw_paths)} files, {sum(os.path.getsize(path) for path in raw_paths.values()) / 1e6:.1f} MB in total")

### Stage 1: read and validate
The file is read one line at a time, as a stream (Notebook 07), so its size does not matter. Every line is checked, and a line that fails is **counted and set aside** with its reason, never silently dropped (Notebook 10).

In [ ]:
def problems_with(order):
    problems = []
    if "order_id" not in order or "time" not in order:
        problems.append("missing order_id or time")
    if not isinstance(order.get("customer"), dict) or "id" not in order.get("customer", {}):
        problems.append("missing customer")
    lines = order.get("lines")
    if not isinstance(lines, list) or not lines:
        problems.append("no order lines")
    elif any(not isinstance(line.get("quantity"), int) or line["quantity"] <= 0 for line in lines):
        problems.append("impossible quantity")
    return problems

def read_and_validate(path):
    good, rejected = [], []
    with open(path, encoding="utf-8") as file:
        for text in file:                                          # one line at a time
            try:
                order = json.loads(text)
            except json.JSONDecodeError:
                rejected.append(("unreadable line", text[:40]))
                continue
            found = problems_with(order)
            if found:
                rejected.append((found[0], order.get("order_id")))
            else:
                good.append(order)
    return good, rejected

good, rejected = read_and_validate(raw_paths[days[0]])
print(f"{len(good):,} orders accepted, {len(rejected)} rejected")
reasons = {}
for reason, detail in rejected:
    reasons[reason] = reasons.get(reason, 0) + 1
print(reasons)

### Stage 2: clean, protect and flatten
Three things happen here.

**Duplicates are removed**, by order identifier, as in Notebook 11.

**Personal data is protected.** An analyst who wants revenue by city has no need of anyone's e-mail address, so we drop it at once: data that is not stored cannot leak. The customer's number we do need, to count returning customers, but not in a form that identifies anyone. We replace it with a **pseudonym**: a hash of the number together with a secret value, the *salt*, known only to the pipeline. The same customer always gets the same pseudonym, so analysis still works, and without the salt the pseudonym cannot be turned back into a customer.

**The nested orders are flattened** into one row per order line, the grain of the fact table of Notebook 04, with the revenue computed and stored at the price of the day.

In [ ]:
SALT = "a-long-secret-value-kept-outside-the-data"

def pseudonym(customer_id):
    return hashlib.sha256(f"{SALT}:{customer_id}".encode()).hexdigest()[:12]

def clean_and_flatten(orders):
    seen, rows, duplicates = set(), [], 0
    for order in orders:
        if order["order_id"] in seen:
            duplicates += 1
            continue
        seen.add(order["order_id"])
        date, clock = order["time"].split("T")
        for number, line in enumerate(order["lines"], start=1):
            rows.append({
                "order_id": order["order_id"], "line": number, "date": date, "hour": int(clock[:2]),
                "customer": pseudonym(order["customer"]["id"]), "city": order["customer"]["city"],
                "title": line["title"], "category": line["category"],
                "quantity": line["quantity"], "revenue": round(line["quantity"] * line["price"], 2),
            })
    return rows, duplicates

rows, duplicates = clean_and_flatten(good)
print(f"{duplicates} duplicate orders removed, {len(rows):,} order lines produced")
print(rows[0])

### Stage 3: store as partitioned Parquet
The clean rows go into the **data lake**: Parquet files in one folder per day, the layout of Notebook 05. As there, the date is not stored inside the file. It is carried by the name of the folder, `date=2024-03-01`. Before writing a day, its folder is emptied. Running the stage a second time therefore **replaces** the day and cannot double it. The stage is idempotent.

In [ ]:
COLUMNS = ["order_id", "line", "hour", "customer", "city", "title", "category", "quantity", "revenue"]      # the date is the folder

def store(rows, day):
    partition = os.path.join(lake_folder, f"date={day.isoformat()}")
    if os.path.exists(partition):
        shutil.rmtree(partition)                                   # replace the day, never add to it
    os.makedirs(partition)
    table = pa.table({column: [row[column] for row in rows] for column in COLUMNS})
    path = os.path.join(partition, "part-0.parquet")
    pq.write_table(table, path)
    return path

stored = store(rows, days[0])
print(os.path.relpath(stored, workspace).replace(os.sep, "/"), f"{os.path.getsize(stored) / 1000:.0f} KB")

### Stage 4: load the warehouse
The managers' reports come from a small warehouse with a star schema (Notebook 04), held here in SQLite. A fact table of sales is surrounded by a date dimension and a product dimension, with an index on the column that reports filter by (Notebook 03).

Loading a day is done inside a **transaction** (Notebook 01): the day's old facts are deleted and the new ones inserted, and either both happen or neither. Again, loading twice gives the same result as loading once.

In [ ]:
warehouse = sqlite3.connect(os.path.join(workspace, "warehouse.db"))
warehouse.executescript("""
    CREATE TABLE IF NOT EXISTS dim_date    (date TEXT PRIMARY KEY, year INTEGER, month INTEGER, weekday TEXT, is_weekend INTEGER);
    CREATE TABLE IF NOT EXISTS dim_product (title TEXT PRIMARY KEY, category TEXT);
    CREATE TABLE IF NOT EXISTS fact_sales  (order_id TEXT, line INTEGER, date TEXT, hour INTEGER, customer TEXT, city TEXT,
                                            title TEXT, quantity INTEGER, revenue REAL);
    CREATE INDEX IF NOT EXISTS sales_by_date ON fact_sales(date);
""")

def load(day):
    table = pq.read_table(lake_folder, filters=[("date", "=", day.isoformat())])       # opens only that day's folder
    records = table.to_pylist()
    with warehouse:                                                # one transaction: all of it, or none of it
        warehouse.execute("DELETE FROM fact_sales WHERE date = ?", (day.isoformat(),))
        warehouse.executemany("INSERT INTO fact_sales VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)",
                              [(r["order_id"], r["line"], r["date"], r["hour"], r["customer"], r["city"],
                                r["title"], r["quantity"], r["revenue"]) for r in records])
        warehouse.execute("INSERT OR REPLACE INTO dim_date VALUES (?, ?, ?, ?, ?)",
                          (day.isoformat(), day.year, day.month, day.strftime("%A"), 1 if day.weekday() >= 5 else 0))
        warehouse.executemany("INSERT OR REPLACE INTO dim_product VALUES (?, ?)",
                              sorted({(r["title"], r["category"]) for r in records}))
    return len(records)

print(f"{load(days[0]):,} facts loaded for {days[0]}")

### Stage 5: check
A pipeline that nobody watches must watch itself. After each day it verifies what it loaded, and it raises an alarm if the share of rejected input is far from the usual.

In [ ]:
def check(day, accepted, rejected_count, duplicates, loaded):
    received = accepted + rejected_count
    in_warehouse = warehouse.execute("SELECT COUNT(*), SUM(revenue) FROM fact_sales WHERE date = ?", (day.isoformat(),)).fetchone()
    in_lake = pq.read_table(lake_folder, filters=[("date", "=", day.isoformat())], columns=["revenue"])
    results = {
        "the lake and the warehouse hold the same number of rows": in_lake.num_rows == in_warehouse[0] == loaded,
        "no order line appears twice": warehouse.execute(
            "SELECT COUNT(*) FROM (SELECT order_id, line FROM fact_sales WHERE date = ? GROUP BY order_id, line HAVING COUNT(*) > 1)",
            (day.isoformat(),)).fetchone()[0] == 0,
        "every quantity and revenue is positive": warehouse.execute(
            "SELECT COUNT(*) FROM fact_sales WHERE date = ? AND (quantity <= 0 OR revenue <= 0)", (day.isoformat(),)).fetchone()[0] == 0,
        "fewer than 5% of the input was rejected": rejected_count / received < 0.05,
    }
    return results

for description, passed in check(days[0], len(good), len(rejected), duplicates, len(rows)).items():
    print(f"  {'PASS' if passed else 'FAIL'}  {description}")

### The pipeline
The five stages, chained into one function that processes one day and returns a short record of what it did. Such records, kept for every run, are how the people responsible for a pipeline know whether last night went well.

In [ ]:
def run_pipeline(day):
    started = time.perf_counter()
    good, rejected = read_and_validate(raw_paths[day])
    rows, duplicates = clean_and_flatten(good)
    stored = store(rows, day)
    loaded = load(day)
    checks = check(day, len(good), len(rejected), duplicates, loaded)
    return {"day": day.isoformat(), "received": len(good) + len(rejected), "rejected": len(rejected),
            "duplicates": duplicates, "lines loaded": loaded, "checks passed": all(checks.values()),
            "seconds": round(time.perf_counter() - started, 2)}

log = [run_pipeline(day) for day in days]

print(f"{'day':<12} {'received':>9} {'rejected':>9} {'duplicates':>11} {'lines loaded':>13} {'checks':>7} {'seconds':>8}")
for entry in log:
    print(f"{entry['day']:<12} {entry['received']:>9,} {entry['rejected']:>9} {entry['duplicates']:>11} "
          f"{entry['lines loaded']:>13,} {'ok' if entry['checks passed'] else 'FAILED':>7} {entry['seconds']:>8}")

### The report
And the reason for all of it: a question a manager can ask in one line of SQL.

In [ ]:
print(f"{'date':<12} {'weekday':<10} {'orders':>7} {'revenue':>12}")
for row in warehouse.execute("""
    SELECT d.date, d.weekday, COUNT(DISTINCT f.order_id), ROUND(SUM(f.revenue), 2)
    FROM fact_sales AS f JOIN dim_date AS d ON d.date = f.date
    GROUP BY d.date ORDER BY d.date
"""):
    print(f"{row[0]:<12} {row[1]:<10} {row[2]:>7,} {row[3]:>12,.2f}")

print()
for row in warehouse.execute("""
    SELECT p.category, ROUND(SUM(f.revenue)) AS revenue, COUNT(DISTINCT f.customer) AS customers
    FROM fact_sales AS f JOIN dim_product AS p ON p.title = f.title
    GROUP BY p.category ORDER BY revenue DESC LIMIT 4
"""):
    print(f"{row[0]:<10} revenue {row[1]:>10,.0f}   from {row[2]:,} different customers")

The count of different customers works although the warehouse holds no customer numbers and no e-mail addresses. The pseudonyms are enough.

## 5. Running it again

Sooner or later a night goes wrong. The file for 3 March turns out to have been incomplete, and a corrected file arrives the next day. Or the machine failed halfway through a load. The operator's remedy must be the simplest possible: **run that day again**. Whether a pipeline tolerates this is the difference between a system that can be operated and one that cannot.

In [ ]:
def totals():
    return warehouse.execute("SELECT COUNT(*), ROUND(SUM(revenue), 2) FROM fact_sales").fetchone()

before = totals()
for attempt in range(3):
    run_pipeline(days[2])                              # the same day, three more times
after = totals()

print("Before:", before)
print("After running 3 March three more times:", after)
print("Unchanged:", before == after)

Nothing changed, because every stage **replaces** its day and never adds to it. Contrast that with the obvious way of writing the load, with a plain `INSERT` and no `DELETE`: each re-run would have added the day's sales again, and the revenue of 3 March would have been four times too high, in a report that looked entirely normal.

The pipeline is also **incremental**. A new day touches only its own partition and its own facts. The cost of a nightly run depends on the size of one day, and stays the same however many years of history have piled up.

In [ ]:
new_day = datetime.date(2024, 3, 6)
raw_paths[new_day] = write_raw_day(new_day)
entry = run_pipeline(new_day)

print(f"{entry['day']}: {entry['lines loaded']:,} lines loaded in {entry['seconds']} seconds, checks {'ok' if entry['checks passed'] else 'FAILED'}")
print("Partitions in the lake:", sorted(os.listdir(lake_folder)))

## 6. The right to be forgotten

A customer writes to the shop and asks for their data to be erased. Under the European Union's data protection regulation, the GDPR, they are generally entitled to that, and the shop has about a month to comply. This is where designs built for convenience meet a requirement they did not plan for.

First we must find the customer. We stored pseudonyms. Because the pipeline holds the salt, it can compute the pseudonym of customer 1234 and look for it.

In [ ]:
target = pseudonym(1234)
found = warehouse.execute("SELECT COUNT(*), ROUND(SUM(revenue), 2) FROM fact_sales WHERE customer = ?", (target,)).fetchone()
print(f"Customer 1234 is stored as {target}: {found[0]} order lines, revenue {found[1]}")

### In the warehouse
A database was built for changing data. Erasing the customer is one statement, in one transaction.

In [ ]:
with warehouse:
    erased = warehouse.execute("DELETE FROM fact_sales WHERE customer = ?", (target,)).rowcount
print(f"Warehouse: {erased} rows deleted with one statement")

### In the lake
The Parquet files are another matter. Notebook 05 warned that such a file cannot be changed in place. To remove a few rows, every file that contains one must be **read, filtered and written out again** in full.

In [ ]:
def erase_from_lake(customer_pseudonym):
    rewritten, removed, examined = 0, 0, 0
    for partition in sorted(os.listdir(lake_folder)):
        path = os.path.join(lake_folder, partition, "part-0.parquet")
        table = pq.ParquetFile(path).read()
        examined += table.num_rows
        keep = pa.array([customer != customer_pseudonym for customer in table["customer"].to_pylist()])
        if table.num_rows - sum(keep.to_pylist()) > 0:
            filtered = table.filter(keep)
            removed += table.num_rows - filtered.num_rows
            pq.write_table(filtered, path)                         # the whole file is written again
            rewritten += 1
    return rewritten, removed, examined

rewritten, removed, examined = erase_from_lake(target)
print(f"Lake: {removed} rows removed. To do it, {examined:,} rows were read and {rewritten} of {len(os.listdir(lake_folder))} files were rewritten.")

To delete a handful of rows, the whole lake had to be read, and most of its files written again. Here that is six small files. In a lake of ten years it is thousands of large ones.

| | Analyse millions of rows | Erase one customer |
|---|---|---|
| Warehouse table (rows, with an index) | workable, slower | one statement, a few rows touched |
| Parquet lake (immutable columnar files) | fast and cheap | every file holding the customer is rewritten |
| Better choice | **the lake** | **the database** |

The immutable files that made analysis fast and failure recovery simple make deletion expensive. Everything in this course has had this shape: a design is good **for something**, and pays for it somewhere else.

There is one more step, and it is the important one. The raw files still hold the customer's real number and their e-mail address. A deletion that leaves them there has deleted nothing. Real systems deal with this by keeping raw files only for a short, fixed time, and by a cleverer use of the idea from Stage 2: if each customer has their **own** secret key and the key is destroyed, every pseudonym of that customer, in every file and every backup, becomes permanently meaningless without a single file being rewritten.

> ⚠️ **Common pitfalls**
>
> - Collecting personal data "in case it is useful". Data you never stored cannot leak, need not be secured, and never has to be erased.
> - Treating a pseudonym as anonymous. Whoever holds the salt can reverse the lookup, and a rare combination of city, date and purchase can identify a person with no key at all. Pseudonymised data is still personal data.
> - Forgetting the copies: raw files, backups, exported spreadsheets, a colleague's laptop. An erasure must reach all of them.
> - Designing for deletion afterwards. Whether a system can forget is decided on the day its storage is laid out.

## 7. The whole course in one picture

The first notebook asked a question: **what changes when the data no longer fits?** Each module met a new way of not fitting, and each answer gave something up to gain something else.

| The data no longer fits... | The answer | What it costs |
|---|---|---|
| in one table (Notebook 01) | several tables, each fact once | questions need joins |
| in the time a query has (03) | an index | slower writes, more storage |
| the questions being asked (04) | a warehouse arranged for reading | data copied and stored twice |
| the shape of the work (05) | columns, compressed | no changing a row in place |
| in a fixed structure (06) | documents, with no enforced schema | every mistake accepted silently |
| on one machine's disk (06) | sharding and replicas | answers that may be stale |
| in memory (07) | streaming, a piece at a time | only small results, or approximate ones |
| in one machine's time (08 to 10) | many machines, pure functions | overhead, and a ceiling set by the sequential part |
| in time itself (11) | windows and watermarks | answers that are never final |

Look down the last column. There is no entry that says "nothing". That is the real content of this course, more than any tool in it. Every one of these techniques is a **trade**, and the skill is to know what is being traded, and to measure whether the trade is worth making for the data and the questions in front of you.

Look also at how few ideas there turned out to be. Sort things so that you can find them. Summarise pieces separately and combine the summaries. Keep each fact in one place, or copy it deliberately and accept the cost. Describe what you want and let the machine decide how. Make every step safe to repeat. Those five appeared at every scale, from a list of five rows to a cluster.

---
## ✏️ Exercises

### Exercise 1 (Amdahl in practice)
A nightly job takes 10 hours on one machine. Measurements show that 30 minutes of it is a step that cannot be parallelised. (a) How long does it take on 8 machines, and on 64? (b) What is the shortest time any number of machines could achieve? (c) The team shortens the sequential step to 5 minutes. Answer (a) and (b) again. Which was the better investment: 64 machines, or fixing the step?

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
def hours(sequential_minutes, total_hours, machines):
    sequential = sequential_minutes / 60
    return sequential + (total_hours - sequential) / machines

for sequential_minutes in [30, 5]:
    print(f"Sequential step of {sequential_minutes} minutes:")
    for machines in [8, 64]:
        print(f"   {machines:>2} machines: {hours(sequential_minutes, 10, machines):.2f} hours")
    print(f"   limit:       {sequential_minutes / 60:.2f} hours\n")
```

*With the 30-minute step, 64 machines give 0.65 hours, and no number of machines can beat half an hour. After the step is shortened, 8 machines already give 1.32 hours and 64 give 0.24, less than half of what was the unbreakable limit before. Shortening the part that cannot be shared moved the ceiling itself. More machines only brought the job closer to a ceiling that stayed where it was.*
</details>

### Exercise 2 (Where is the break-even?)
With the illustrative prices of Section 3, find the nightly workload, in core-hours, at which a 16-core always-on machine and a cluster rented by the hour cost the same. Then express that workload as the **utilisation** of the always-on machine. What rule of thumb follows?

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
fixed = always_on(16)
break_even = fixed / (30 * PRICE_PER_CORE_HOUR)             # core-hours per night at which rented_nightly equals fixed

print(f"Always-on machine: {fixed:,.0f} a month")
print(f"Break-even: {break_even:.0f} core-hours a night")
print(f"Capacity of the machine: {16 * 24} core-hours a day, so the break-even utilisation is {break_even / (16 * 24):.0%}")
```

*The reserved machine costs half as much per core-hour, so it becomes the cheaper choice once it would be busy half of the time. In general, the break-even utilisation equals the ratio of the two prices. The rule of thumb: a machine that would be busy most of the time should be reserved, and work that comes in bursts should be rented by the hour.*
</details>

### Exercise 3 (A new check)
Add a check to the pipeline: on any day, no single hour should account for more than 15% of the day's order lines. (A spike like that would suggest a fault upstream, or a duplicated batch.) Write `busiest_hour_share(day)` using a query on `fact_sales`, and run it for every day in the warehouse.

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
def busiest_hour_share(day):
    rows = warehouse.execute("SELECT hour, COUNT(*) FROM fact_sales WHERE date = ? GROUP BY hour", (day,)).fetchall()
    total = sum(count for hour, count in rows)
    hour, count = max(rows, key=lambda row: row[1])
    return hour, count / total

for (day,) in warehouse.execute("SELECT date FROM dim_date ORDER BY date").fetchall():
    hour, share = busiest_hour_share(day)
    print(f"{day}: busiest hour {hour:02d}:00 with {share:.1%} of the lines   {'PASS' if share <= 0.15 else 'FAIL'}")
```

*With 24 hours a perfectly even day would give about 4% to each. A check like this knows nothing about what went wrong. It only notices that the data has an unusual shape, which is very often the first visible sign of a fault. The index on `date` makes each of these queries a quick search.*
</details>

### Exercise 4 (A corrected file)
The file for 2 March is replaced by a corrected one containing 3,400 orders in place of 3,000. Write the new file with `write_raw_day(days[1], orders=3400)`, run the pipeline for that day, and show that the warehouse now holds the corrected figures for 2 March while every other day is unchanged.

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
def per_day():
    return dict(warehouse.execute("SELECT date, COUNT(DISTINCT order_id) FROM fact_sales GROUP BY date").fetchall())

before = per_day()
raw_paths[days[1]] = write_raw_day(days[1], orders=3400)
entry = run_pipeline(days[1])
after = per_day()

for day in sorted(after):
    marker = "   <- corrected" if before[day] != after[day] else ""
    print(f"{day}: {before[day]:>5,} orders before, {after[day]:>5,} after{marker}")
print("Checks passed:", entry["checks passed"])
```

*Only the day that was run again changed. Because each stage replaces the whole of its day, a correction needs no special procedure: put the right file in place and run the day again. A pipeline that could only add would have needed someone to work out, by hand, exactly what to remove first.*
</details>

### Exercise 5 (Erase, and prove it, a little harder)
Customer 777 asks to be erased. (a) Erase them from the warehouse and from the lake. (b) Write a function `still_present(customer_id)` that searches the warehouse **and** the lake for the customer's pseudonym and returns how many rows it finds in each. Use it to prove the erasure worked. (c) Now search the **raw** files for the customer's e-mail address, `customer777@example.com`. What do you find, and what should be done about it?

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
def still_present(customer_id):
    key = pseudonym(customer_id)
    in_warehouse = warehouse.execute("SELECT COUNT(*) FROM fact_sales WHERE customer = ?", (key,)).fetchone()[0]
    in_lake = pq.read_table(lake_folder, columns=["customer"], filters=[("customer", "=", key)]).num_rows
    return in_warehouse, in_lake

print("Before:", still_present(777))

key = pseudonym(777)
with warehouse:
    warehouse.execute("DELETE FROM fact_sales WHERE customer = ?", (key,))
erase_from_lake(key)
print("After: ", still_present(777))

in_raw = 0
for path in raw_paths.values():
    with open(path, encoding="utf-8") as file:
        in_raw += sum(1 for line in file if "customer777@example.com" in line)
print(f"Lines in the raw files that still contain the customer's e-mail address: {in_raw}")
```

*The warehouse and the lake are clean, and the raw files still hold the customer's e-mail address on every one of their orders. The erasure is not complete until the raw files are dealt with as well: by rewriting them, or far better by a rule that raw files are deleted a few days after they have been processed. This is the commonest way for a deletion to fail in practice. The system that was cleaned is the one people think of, and the copy that was forgotten is the one that matters.*
</details>

## 🔑 Key takeaways

- The **cloud** is rented computing: elastic, paid for by use, with storage kept separate from the machines that compute on it.
- **Amdahl's law**: the part of a job that cannot be parallelised sets a hard limit on the speedup. Shrink it before adding machines.
- Cost depends on **utilisation**. Rent by the hour for uneven work, and permanently for steady work. In services that charge by the byte read, the **file format** is the largest lever.
- A **pipeline** chains validation, cleaning, storage, loading and checking. It should count what it rejects and verify what it produces.
- Make every stage **idempotent** (safe to run again) and **incremental** (touching only what is new).
- Protect personal data from the first stage: store less, **pseudonymise** what you keep, and design for **erasure** from the start.
- Every technique in this course is a **trade**. Know what is being traded, and measure.

### 🎓 Congratulations: you have completed the entire course!

You began by keeping five sales in one list, and saw it contradict itself within a dozen lines. You finish with a pipeline that takes in messy files every night, protects the people in them, stores them in a form that is cheap to analyse, loads a warehouse, checks its own work, survives being run twice, and can forget a customer on request.

On the way you built, by hand, a hash join, an index with binary search, a star schema, dictionary and run-length encoders, a document store, a consistent-hashing ring, a replicated store, a streaming reader, a reservoir sampler, an external merge sort, a MapReduce engine with combiners and failure recovery, and windowing with watermarks. You then used the professional tools that do each of these things, and found nothing in them that you had not already built in miniature.

You also measured, every time, and several times the measurement surprised you: a scan beat an index, a plain loop beat a cluster engine, a single machine beat Spark. Keep that habit above all the others. The field is full of confident advice about what is fast and what scales. A stopwatch and a small experiment will tell you more.

The questions to carry with you are few. How big is the data, really? How is it read, and how does it change? What must never be wrong? What happens when this runs twice, or half-runs and stops? And what, exactly, am I giving up to get this?

Thank you for working through these notebooks. The best next step is to take some data you care about, and build something for it.

---
*End of Module D: and the end of the course.*

---
*© Ioannis Tsioulis. *From One Table to a Cluster: Working with Data at Scale*. Prepared for educational use.*